In [2]:
from openai import OpenAI
from google.colab import userdata
import os
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

## 시스템 프롬프트

OUTPUT_PATH
llm_feature.csv -> 본인만의 이름으로 설정. 대신 끝에 .csv는 붙이는 편이 좋다!

In [ ]:
# API 키 설정
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# 경로 설정
TRANSCRIPT_DIR = "/content/drive/MyDrive/26-1_URP/transcripts/"
OUTPUT_PATH = "/content/drive/MyDrive/26-1_URP/features/llm_features.csv"


In [ ]:
# 프롬프트 #원하는 걸로 수정해도 됨
system_prompt = """Role
Act like you are a psychologist, an emotion regulation researcher, and a mental health professional.

Instruction
The following is a transcript of an interview between an interviewer and a participant, which is presented inside the triple backticks below.
The interview concerns a stressful or emotionally significant situation experienced by the participant. Determine the extent to which the participant demonstrates the following emotion regulation strategies from the Emotion Regulation Questionnaire (ERQ), which measures emotion regulation tendencies using a 7-point scale:

"When I want to feel more positive emotion (such as joy or amusement), I change what I'm thinking about."
"I keep my emotions to myself."
"When I want to feel less negative emotion (such as sadness or anger), I change what I'm thinking about."
"When I am feeling positive emotions, I am careful not to express them."
"When I'm faced with a stressful situation, I make myself think about it in a way that helps me stay calm."
"I control my emotions by not expressing them."
"When I want to feel more positive emotion, I change the way I'm thinking about the situation."
"I control my emotions by changing the way I think about the situation I'm in."
"When I am feeling negative emotions, I make sure not to express them."
"When I want to feel less negative emotion, I change the way I'm thinking about the situation."

Cognitive reappraisal involves changing the interpretation of a situation in order to alter its emotional impact. Expressive suppression involves inhibiting emotional expression.

The participant's responses should be evaluated using the following scale:
1: Strongly disagree
2: Disagree
3: Slightly disagree
4: Neutral
5: Slightly agree
6: Agree
7: Strongly agree

Language Note
The transcript is written in Korean.
Evaluate based on the Korean text as-is, without translating it first.
Be especially attentive to indirect expressions of suppression, as Korean speakers may express emotional concealment implicitly rather than explicitly.

Steps
For each of the items in the ERQ questionnaire, determine how strongly the participant demonstrates the corresponding emotion regulation tendency.
Start by explaining where in the transcript the participant indicated that particular emotion regulation tendency from the ERQ items (1-2 sentences per item). Then give a score from 1 to 7 based on the item-specific scoring rubric.
Base scores only on explicit or strongly implied evidence from the transcript, and distinguish emotional experience from deliberate emotion regulation.

Item-Specific Scoring Rubrics
Note: Choose any score from 1 to 7, not just 1, 4, or 7. The anchors below are reference points only.

Item 1
1 = No attempt to alter thoughts to increase positive emotion
4 = Some evidence of redirecting attention toward positive thoughts
7 = Detailed and effective cognitive shifting that clearly increased positive emotion

Item 2
1 = Emotion openly expressed
4 = Moderate tendency to hide emotions
7 = Explicit and consistent concealment of emotions from others

Item 3
1 = No attempt to change thoughts to reduce negative emotion
4 = Some attempt to redirect negative thinking
7 = Explicit and sustained thought-changing process that reduced negative emotion

Item 4
1 = Positive emotions expressed freely
4 = Moderate caution about showing positive feelings
7 = Strong and persistent concealment of positive emotional expression

Item 5
1 = No cognitive calming strategy
4 = Some calming reinterpretation
7 = Highly effective cognitive reframing that reduced stress

Item 6
1 = Emotion regulated through expression rather than suppression
4 = Moderate reliance on non-expression
7 = Explicit belief that emotions should be controlled through non-expression

Item 7
1 = No reinterpretation attempt
4 = Some perspective change with limited detail
7 = Detailed and emotionally transformative reappraisal

Item 8
1 = No evidence of cognitive emotion regulation
4 = Moderate reinterpretation tendency
7 = Explicit reliance on cognitive reframing as a primary emotional control strategy

Item 9
1 = Negative emotions openly expressed
4 = Moderate effort to hide negative emotion
7 = Explicit and persistent concealment of negative emotional expression

Item 10
1 = No reinterpretation effort
4 = Some reinterpretation of the stressful situation
7 = Detailed reinterpretation producing emotional relief or recovery

End Goal
In the end, I want you to give me a score between 1 and 7 for each of the ERQ questions.
Additionally, calculate:
the mean Cognitive Reappraisal score (Items 1, 3, 5, 7, 8, and 10)
the mean Expressive Suppression score (Items 2, 4, 6, and 9)

Output Format
First, provide brief evidence for each item (1-2 sentences each).
Then output ONLY the following JSON at the very end, with no extra text after it:

{
  "item1": ,
  "item2": ,
  "item3": ,
  "item4": ,
  "item5": ,
  "item6": ,
  "item7": ,
  "item8": ,
  "item9": ,
  "item10": ,
  "cognitive_reappraisal_mean": ,
  "expressive_suppression_mean":
}
"""

# 실제 존재하는 참가자 번호 추출
pids = set()
for f in os.listdir(TRANSCRIPT_DIR):
    if f.endswith(".txt"):
        pid = int(f.split("_")[1])
        pids.add(pid)

print(f"총 참가자 수: {len(pids)}")

# 메인 루프
results = []

for pid in sorted(pids):
    txt_path = os.path.join(TRANSCRIPT_DIR, f"interview_{pid}_2.txt")  # 2번만 (stress 조건)

    if not os.path.exists(txt_path):
        print(f"[없음] {pid}번 전사본 없음")
        continue

    # 전사본 읽기
    with open(txt_path, "r", encoding="utf-8") as f:
        transcript = f.read()

    # GPT 호출
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"```{transcript}```"}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content
        results.append({
            "participant_id": pid,
            "gpt_output": result_text
        })
        print(f"[완료] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 저장
df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")
print(f"완료된 참가자 수: {len(df)}")

총 참가자 수: 75
[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료: /content/drive/MyDrive/26-1_URP/features/llm_features.csv
완료된 참가자 수: 75


In [ ]:
import pandas as pd
import json
import re

# 저장된 CSV 불러오기 #llm_feature.csv -> 파싱하고자 하는 파일명으로 이름 바꾸기
df = pd.read_csv("/content/drive/MyDrive/26-1_URP/features/llm_features.csv")

parsed_results = []

for _, row in df.iterrows():
    pid = row['participant_id']
    gpt_output = row['gpt_output']

    try:
        # "" → " 치환 후 JSON 추출
        cleaned = gpt_output.replace('""', '"')
        json_match = re.search(r'\{[^{}]+\}', cleaned, re.DOTALL)

        if json_match:
            scores = json.loads(json_match.group())
            scores['participant_id'] = pid
            parsed_results.append(scores)
            print(f"[완료] {pid}번")
        else:
            print(f"[JSON 없음] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 깔끔한 CSV로 저장
df_parsed = pd.DataFrame(parsed_results)

# participant_id를 맨 앞으로
cols = ['participant_id'] + [c for c in df_parsed.columns if c != 'participant_id']
df_parsed = df_parsed[cols]

df_parsed.to_csv(
    "/content/drive/MyDrive/26-1_URP/features/llm_features_parsed.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n저장 완료!")
print(f"참가자 수: {len(df_parsed)}")
print(df_parsed.head())

[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료!
참가자 수: 75
   participant_id  item1  item2  item3  item4  item5  item6  item7  item8  \
0               2      1      4      2      4      5      6      2      2   
1               3      2      4      4      4      4      4      3      4   
2               4      1      1      1      1      1      1      1      1   
3

## 한국어 프롬프트

In [ ]:
# 경로 설정
TRANSCRIPT_DIR = "/content/drive/MyDrive/26-1_URP/transcripts/"
OUTPUT_PATH = "/content/drive/MyDrive/26-1_URP/features/llm_features_kor.csv"

In [ ]:
# 프롬프트
system_prompt = """역할
당신은 심리학자이자 정서조절 연구자, 정신건강 전문가입니다.

지시사항
아래 세 개의 백틱(```) 안에 있는 텍스트는 면접자와 참가자 간의 인터뷰 전사본입니다.
이 인터뷰는 참가자가 경험한 스트레스 상황 또는 정서적으로 중요한 사건에 관한 것입니다.
아래의 정서조절 전략 척도(ERQ)에 해당하는 경향을 참가자가 얼마나 보이는지 평가해주세요.
ERQ는 7점 척도로 정서조절 경향을 측정합니다:

"나는 더 긍정적인 감정(예: 기쁨, 즐거움)을 느끼고 싶을 때, 생각하는 내용을 바꾼다."
"나는 감정을 혼자 속으로 삭인다."
"나는 덜 부정적인 감정(예: 슬픔, 분노)을 느끼고 싶을 때, 생각하는 내용을 바꾼다."
"나는 긍정적인 감정을 느낄 때, 그것을 표현하지 않으려고 조심한다."
"나는 스트레스 상황에 처했을 때, 침착함을 유지하는 데 도움이 되는 방식으로 그 상황을 생각하려고 노력한다."
"나는 감정을 표현하지 않음으로써 감정을 통제한다."
"나는 더 긍정적인 감정을 느끼고 싶을 때, 상황을 바라보는 방식을 바꾼다."
"나는 내가 처한 상황을 생각하는 방식을 바꿈으로써 감정을 통제한다."
"나는 부정적인 감정을 느낄 때, 그것을 표현하지 않으려고 노력한다."
"나는 덜 부정적인 감정을 느끼고 싶을 때, 상황을 바라보는 방식을 바꾼다."

인지적 재평가(Cognitive Reappraisal)는 상황의 해석을 바꿔 정서적 영향을 변화시키는 전략입니다.
표현 억제(Expressive Suppression)는 감정 표현을 억제하는 전략입니다.

참가자의 응답은 아래 척도로 평가합니다:
1: 전혀 그렇지 않다
2: 그렇지 않다
3: 약간 그렇지 않다
4: 보통이다
5: 약간 그렇다
6: 그렇다
7: 매우 그렇다

언어 유의사항
전사본은 한국어로 작성되어 있습니다.
한국어 화자는 정서조절을 간접적으로 표현하는 경우가 많습니다.
예를 들어 "스스로를 다독였다", "그냥 넘기려 했다", "마음을 고쳐먹었다", "별거 아니라고 생각했다" 같은 표현도 인지적 재평가나 표현 억제의 증거로 볼 수 있습니다.
명시적인 표현이 없더라도 맥락상 강하게 암시되는 경우 점수에 반영하세요.

평가 절차
ERQ 각 항목에 대해 참가자가 해당 정서조절 경향을 얼마나 보이는지 판단하세요.
전사본에서 해당 항목의 근거가 되는 부분을 1~2문장으로 설명한 뒤, 항목별 채점 기준에 따라 1~7점을 부여하세요.
전사본에서 명시적으로 드러나거나 강하게 암시된 근거에만 기반하여 점수를 매기세요.
정서적 경험과 의도적인 정서조절을 구분하세요.

항목별 채점 기준
1, 4, 7은 기준점일 뿐이며, 1~7 중 어느 점수든 선택할 수 있습니다.

항목 1
1 = 긍정 감정을 높이기 위한 사고 전환 시도 없음
4 = 긍정적 생각으로 주의를 돌리려는 일부 증거 있음
7 = 긍정 감정을 명확히 높인 상세하고 효과적인 사고 전환

항목 2
1 = 감정을 공개적으로 표현함
4 = 감정을 숨기려는 중간 정도의 경향
7 = 타인에게 감정을 명시적이고 일관되게 숨김

항목 3
1 = 부정 감정을 줄이기 위한 사고 전환 시도 없음
4 = 부정적 사고를 전환하려는 일부 시도
7 = 부정 감정을 줄인 명시적이고 지속적인 사고 전환 과정

항목 4
1 = 긍정 감정을 자유롭게 표현함
4 = 긍정적 감정 표현에 중간 정도의 주의를 기울임
7 = 긍정적 감정 표현을 강하고 지속적으로 억제함

항목 5
1 = 인지적 진정 전략 없음
4 = 일부 진정적 재해석
7 = 스트레스를 효과적으로 감소시킨 고도의 인지적 재구성

항목 6
1 = 표현을 통해 감정을 조절함 (억제 없음)
4 = 비표현에 중간 정도 의존
7 = 감정을 표현하지 않음으로써 통제해야 한다는 명시적 신념

항목 7
1 = 재해석 시도 없음
4 = 제한적 세부사항을 가진 일부 관점 변화
7 = 상세하고 정서적으로 변화를 가져온 재평가

항목 8
1 = 인지적 정서조절의 증거 없음
4 = 중간 정도의 재해석 경향
7 = 인지적 재구성을 주요 정서 통제 전략으로 명시적으로 사용

항목 9
1 = 부정 감정을 공개적으로 표현함
4 = 부정 감정을 숨기려는 중간 정도의 노력
7 = 부정적 감정 표현을 명시적이고 지속적으로 억제함

항목 10
1 = 재해석 노력 없음
4 = 스트레스 상황에 대한 일부 재해석
7 = 정서적 회복이나 안도감을 가져온 상세한 재해석

최종 목표
ERQ 각 항목에 대해 1~7점을 부여하세요.
추가로 아래를 계산하세요:
인지적 재평가 평균 점수 (항목 1, 3, 5, 7, 8, 10의 평균)
표현 억제 평균 점수 (항목 2, 4, 6, 9의 평균)

출력 형식
각 항목에 대해 근거를 1~2문장으로 설명한 뒤,
아래 JSON을 마지막에 출력하세요. JSON 이후에는 아무 텍스트도 추가하지 마세요:

{
  "item1": ,
  "item2": ,
  "item3": ,
  "item4": ,
  "item5": ,
  "item6": ,
  "item7": ,
  "item8": ,
  "item9": ,
  "item10": ,
  "cognitive_reappraisal_mean": ,
  "expressive_suppression_mean":
}
"""

# 실제 존재하는 참가자 번호 추출
pids = set()
for f in os.listdir(TRANSCRIPT_DIR):
    if f.endswith(".txt"):
        pid = int(f.split("_")[1])
        pids.add(pid)

print(f"총 참가자 수: {len(pids)}")

# 메인 루프
results = []

for pid in sorted(pids):
    txt_path = os.path.join(TRANSCRIPT_DIR, f"interview_{pid}_2.txt")  # 2번만 (stress 조건)

    if not os.path.exists(txt_path):
        print(f"[없음] {pid}번 전사본 없음")
        continue

    # 전사본 읽기
    with open(txt_path, "r", encoding="utf-8") as f:
        transcript = f.read()

    # GPT 호출
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"```{transcript}```"}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content
        results.append({
            "participant_id": pid,
            "gpt_output": result_text
        })
        print(f"[완료] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 저장
df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")
print(f"완료된 참가자 수: {len(df)}")

총 참가자 수: 75
[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료: /content/drive/MyDrive/26-1_URP/features/llm_features_kor.csv
완료된 참가자 수: 75


In [ ]:
import pandas as pd
import json
import re

# 저장된 CSV 불러오기
df = pd.read_csv("/content/drive/MyDrive/26-1_URP/features/llm_features_kor.csv")

parsed_results = []

for _, row in df.iterrows():
    pid = row['participant_id']
    gpt_output = row['gpt_output']

    try:
        # "" → " 치환 후 JSON 추출
        cleaned = gpt_output.replace('""', '"')
        json_match = re.search(r'\{[^{}]+\}', cleaned, re.DOTALL)

        if json_match:
            scores = json.loads(json_match.group())
            scores['participant_id'] = pid
            parsed_results.append(scores)
            print(f"[완료] {pid}번")
        else:
            print(f"[JSON 없음] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 깔끔한 CSV로 저장
df_parsed = pd.DataFrame(parsed_results)

# participant_id를 맨 앞으로
cols = ['participant_id'] + [c for c in df_parsed.columns if c != 'participant_id']
df_parsed = df_parsed[cols]

df_parsed.to_csv(
    "/content/drive/MyDrive/26-1_URP/features/llm_features_kor_parsed.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n저장 완료!")
print(f"참가자 수: {len(df_parsed)}")
print(df_parsed.head())

[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료!
참가자 수: 75
   participant_id  item1  item2  item3  item4  item5  item6  item7  item8  \
0               2      1      6      1      6      4      6      1      1   
1               3      5      6      5      5      5      6      5      5   
2               4      5      4      4      4      5      4      5      5   
3

## 예린이 프롬프트

In [ ]:
# 경로 설정
TRANSCRIPT_DIR = "/content/drive/MyDrive/26-1_URP/transcripts/"
OUTPUT_PATH = "/content/drive/MyDrive/26-1_URP/features/llm_features_yelin.csv"


# 프롬프트
system_prompt = """당신은 감정조절 전략을 평가하는 심리학 기반 채점자입니다.
아래 응답을 읽고 인지적 재평가와 억제 전략을 각각 1~7점 리커트 척도로 평가하세요.

인지적 재평가 점수:
1점 = 재평가가 전혀 없음. 상황의 의미를 다르게 해석하려는 시도가 없음.
2점 = 재평가 가능성이 매우 약함. 긍정적 표현은 있으나 구체적 관점 전환이 없음.
3점 = 막연한 긍정적 사고가 있음. 하지만 어떻게 의미를 바꿨는지 불분명함.
4점 = 상황을 다르게 보려는 시도가 어느 정도 있음. 다만 설명이 짧거나 구체성이 부족함.
5점 = 명확한 관점 전환 또는 의미 재해석이 있음.
6점 = 적극적인 의미 재구성이 있으며 감정 조절 의도가 뚜렷함.
7점 = 매우 명확하고 구체적인 의미 재구성이 있고, 그 결과 감정이 완화되거나 변화한 내용까지 나타남.

억제 점수:
1점 = 억제가 전혀 없음. 감정을 숨기거나 표현을 억누른 내용이 없음.
2점 = 억제 가능성이 매우 약함. 감정을 드러내지 않았을 가능성은 있으나 불명확함.
3점 = 감정을 참으려는 표현이 약하게 있음. 하지만 구체적 행동은 부족함.
4점 = 감정을 겉으로 드러내지 않으려는 시도가 어느 정도 나타남.
5점 = 감정 표현을 의도적으로 억제한 것이 명확함.
6점 = 강한 감정을 느꼈지만 표현하지 않으려는 의도와 행동이 뚜렷함.
7점 = 강한 감정을 철저히 숨기거나 아무렇지 않은 척하는 등 억제가 매우 명확함.

[세부 판단 기준]
인지적 재평가는 다음 요소가 많고 구체적일수록 높은 점수를 부여하세요:
- 상황을 다른 관점에서 해석함
- 타인의 입장이나 상황을 고려함
- 부정적 경험을 배움, 성장, 기회로 재구성함
- 사건의 원인이나 의미를 다르게 생각함
- 재해석 후 감정이 완화되거나 변화함

억제는 다음 요소가 많고 구체적일수록 높은 점수를 부여하세요:
- 부정적 감정을 느꼈음이 명확함
- 감정을 말, 표정, 행동으로 드러내지 않음
- 티 내지 않음, 참음, 숨김, 괜찮은 척함
- 일부러 감정 표현을 억누름
- 강한 감정을 느꼈지만 겉으로는 다른 모습을 보임

[구분 규칙]
- "다르게 생각했다", "이해하려 했다", "배움이라고 생각했다"처럼 상황의 의미를 바꾸면 인지적 재평가로 평가하세요.
- "참았다", "티 내지 않았다", "아무렇지 않은 척했다"처럼 감정 표현을 숨기면 억제로 평가하세요.
- 하나의 응답에 두 전략이 모두 있으면 둘 다 점수를 부여하고 main_strategy는 mixed로 표시하세요.
- 단순히 "생각했다"라는 단어만으로 재평가로 판단하지 마세요.
- 단순히 "조용히 있었다"만으로 높은 억제 점수를 주지 마세요. 감정을 숨기거나 표현을 억제한 맥락이 필요합니다.

[언어 유의사항]
전사본은 한국어로 작성되어 있습니다.
한국어 화자는 정서조절을 간접적으로 표현하는 경우가 많습니다.
"스스로를 다독였다", "별거 아니라고 생각했다", "그냥 넘기려 했다", "마음을 고쳐먹었다" 같은
간접적 표현도 각 전략의 근거로 볼 수 있습니다.

[confidence 기준]
high = 근거가 명확하고 전략이 뚜렷하게 드러남
medium = 근거가 있으나 해석의 여지가 있음
low = 근거가 불명확하거나 두 전략의 구분이 어려움

[출력 형식]
각 전략에 대해 근거를 1~2문장으로 설명한 뒤,
아래 JSON을 마지막에 출력하세요. JSON 이후에는 아무 텍스트도 추가하지 마세요:

{
  "cognitive_reappraisal": ,
  "expressive_suppression": ,
  "main_strategy": ,
  "confidence": ,
  "confidence_reason":
}
"""

# 실제 존재하는 참가자 번호 추출
pids = set()
for f in os.listdir(TRANSCRIPT_DIR):
    if f.endswith(".txt"):
        pid = int(f.split("_")[1])
        pids.add(pid)

print(f"총 참가자 수: {len(pids)}")

# 메인 루프
results = []

for pid in sorted(pids):
    txt_path = os.path.join(TRANSCRIPT_DIR, f"interview_{pid}_2.txt")  # 2번만 (stress 조건)

    if not os.path.exists(txt_path):
        print(f"[없음] {pid}번 전사본 없음")
        continue

    # 전사본 읽기
    with open(txt_path, "r", encoding="utf-8") as f:
        transcript = f.read()

    # GPT 호출
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"```{transcript}```"}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content
        results.append({
            "participant_id": pid,
            "gpt_output": result_text
        })
        print(f"[완료] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 저장
df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")
print(f"완료된 참가자 수: {len(df)}")

총 참가자 수: 75
[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료: /content/drive/MyDrive/26-1_URP/features/llm_features_yelin.csv
완료된 참가자 수: 75


In [ ]:
import pandas as pd
import json
import re

# CSV 불러오기
df = pd.read_csv("/content/drive/MyDrive/26-1_URP/features/llm_features_yelin.csv")

parsed_results = []

for _, row in df.iterrows():
    pid = row['participant_id']
    gpt_output = row['gpt_output']

    try:
        # "" → " 치환 후 JSON 추출
        cleaned = gpt_output.replace('""', '"')
        json_match = re.search(r'\{[^{}]+\}', cleaned, re.DOTALL)

        if json_match:
            scores = json.loads(json_match.group())
            scores['participant_id'] = pid
            parsed_results.append(scores)
            print(f"[완료] {pid}번")
        else:
            print(f"[JSON 없음] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 데이터프레임 변환
df_parsed = pd.DataFrame(parsed_results)

# participant_id 맨 앞으로
cols = ['participant_id'] + [c for c in df_parsed.columns if c != 'participant_id']
df_parsed = df_parsed[cols]

df_parsed.to_csv(
    "/content/drive/MyDrive/26-1_URP/features/llm_features_yelin_parsed.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n저장 완료!")
print(f"참가자 수: {len(df_parsed)}")
print(df_parsed.head())

[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료!
참가자 수: 75
   participant_id  cognitive_reappraisal  expressive_suppression  \
0               2                      5                       6   
1               3                      3                       5   
2               4                      5                       5   
3               5                    

In [6]:
# API 키 설정
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# 경로 설정
TRANSCRIPT_DIR = "/content/drive/MyDrive/26-1_URP/transcripts/"
OUTPUT_PATH = "/content/drive/MyDrive/26-1_URP/features/llm_features_deleted.csv"

In [8]:
system_prompt = """
역할
당신은 심리학자이자 정서조절 연구자, 정신건강 전문가입니다.

지시사항
아래 세 개의 백틱(```) 안에 있는 텍스트는 면접자와 참가자 간의 인터뷰 전사본입니다.
이 인터뷰는 참가자가 경험한 스트레스 상황 또는 정서적으로 중요한 사건에 관한 것입니다.
아래의 정서조절 전략 척도(ERQ)에 해당하는 경향을 참가자가 얼마나 보이는지 평가해주세요.
ERQ는 7점 척도로 정서조절 경향을 측정합니다:

"나는 더 긍정적인 감정(예: 기쁨, 즐거움)을 느끼고 싶을 때, 생각하는 내용을 바꾼다."
"나는 감정을 혼자 속으로 삭인다."
"나는 덜 부정적인 감정(예: 슬픔, 분노)을 느끼고 싶을 때, 생각하는 내용을 바꾼다."
"나는 긍정적인 감정을 느낄 때, 그것을 표현하지 않으려고 조심한다."
"나는 스트레스 상황에 처했을 때, 침착함을 유지하는 데 도움이 되는 방식으로 그 상황을 생각하려고 노력한다."
"나는 감정을 표현하지 않음으로써 감정을 통제한다."
"나는 더 긍정적인 감정을 느끼고 싶을 때, 상황을 바라보는 방식을 바꾼다."
"나는 내가 처한 상황을 생각하는 방식을 바꿈으로써 감정을 통제한다."
"나는 부정적인 감정을 느낄 때, 그것을 표현하지 않으려고 노력한다."
"나는 덜 부정적인 감정을 느끼고 싶을 때, 상황을 바라보는 방식을 바꾼다."

인지적 재평가(Cognitive Reappraisal)는 상황의 해석을 바꿔 정서적 영향을 변화시키는 전략입니다.
표현 억제(Expressive Suppression)는 감정 표현을 억제하는 전략입니다.

참가자의 응답은 아래 척도로 평가합니다:
1: 전혀 그렇지 않다
2: 그렇지 않다
3: 약간 그렇지 않다
4: 보통이다
5: 약간 그렇다
6: 그렇다
7: 매우 그렇다

언어 유의사항
전사본은 한국어로 작성되어 있습니다.
한국어 화자는 정서조절을 간접적으로 표현하는 경우가 많습니다.
예를 들어 "스스로를 다독였다", "그냥 넘기려 했다", "마음을 고쳐먹었다", "별거 아니라고 생각했다" 같은 표현도 인지적 재평가나 표현 억제의 증거로 볼 수 있습니다.
명시적인 표현이 없더라도 맥락상 강하게 암시되는 경우 점수에 반영하세요.

평가 절차
ERQ 각 항목에 대해 참가자가 해당 정서조절 경향을 얼마나 보이는지 판단하세요.
전사본에서 해당 항목의 근거가 되는 부분을 1~2문장으로 설명한 뒤, 항목별 채점 기준에 따라 1~7점을 부여하세요.
전사본에서 명시적으로 드러나거나 강하게 암시된 근거에만 기반하여 점수를 매기세요.
정서적 경험과 의도적인 정서조절을 구분하세요.

항목별 채점 기준
1, 4, 7은 기준점일 뿐이며, 1~7 중 어느 점수든 선택할 수 있습니다.

항목 2
1 = 감정을 공개적으로 표현함
4 = 감정을 숨기려는 중간 정도의 경향
7 = 타인에게 감정을 명시적이고 일관되게 숨김

항목 3
1 = 부정 감정을 줄이기 위한 사고 전환 시도 없음
4 = 부정적 사고를 전환하려는 일부 시도
7 = 부정 감정을 줄인 명시적이고 지속적인 사고 전환 과정

항목 6
1 = 표현을 통해 감정을 조절함 (억제 없음)
4 = 비표현에 중간 정도 의존
7 = 감정을 표현하지 않음으로써 통제해야 한다는 명시적 신념

항목 8
1 = 인지적 정서조절의 증거 없음
4 = 중간 정도의 재해석 경향
7 = 인지적 재구성을 주요 정서 통제 전략으로 명시적으로 사용

항목 9
1 = 부정 감정을 공개적으로 표현함
4 = 부정 감정을 숨기려는 중간 정도의 노력
7 = 부정적 감정 표현을 명시적이고 지속적으로 억제함

항목 10
1 = 재해석 노력 없음
4 = 스트레스 상황에 대한 일부 재해석
7 = 정서적 회복이나 안도감을 가져온 상세한 재해석

최종 목표
ERQ 각 항목에 대해 1~7점을 부여하세요.
추가로 아래를 계산하세요:
인지적 재평가 평균 점수 (항목  3, 8, 10의 평균)
표현 억제 평균 점수 (항목 2, 6, 9의 평균)

출력 형식
각 항목에 대해 근거를 1~2문장으로 설명한 뒤,
아래 JSON을 마지막에 출력하세요. JSON 이후에는 아무 텍스트도 추가하지 마세요:

{
  "item2": ,
  "item3": ,
  "item6": ,
  "item8": ,
  "item9": ,
  "item10": ,
  "cognitive_reappraisal_mean": ,
  "expressive_suppression_mean":
}
"""

# 실제 존재하는 참가자 번호 추출
pids = set()
for f in os.listdir(TRANSCRIPT_DIR):
    if f.endswith(".txt"):
        pid = int(f.split("_")[1])
        pids.add(pid)

print(f"총 참가자 수: {len(pids)}")

# 메인 루프
results = []

for pid in sorted(pids):
    txt_path = os.path.join(TRANSCRIPT_DIR, f"interview_{pid}_2.txt")  # 2번만 (stress 조건)

    if not os.path.exists(txt_path):
        print(f"[없음] {pid}번 전사본 없음")
        continue

    # 전사본 읽기
    with open(txt_path, "r", encoding="utf-8") as f:
        transcript = f.read()

    # GPT 호출
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"```{transcript}```"}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content
        results.append({
            "participant_id": pid,
            "gpt_output": result_text
        })
        print(f"[완료] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 저장
df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")
print(f"완료된 참가자 수: {len(df)}")

총 참가자 수: 75
[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료: /content/drive/MyDrive/26-1_URP/features/llm_features_deleted.csv
완료된 참가자 수: 75


In [9]:
import pandas as pd
import json
import re

# CSV 불러오기
df = pd.read_csv("/content/drive/MyDrive/26-1_URP/features/llm_features_deleted.csv")

parsed_results = []

for _, row in df.iterrows():
    pid = row['participant_id']
    gpt_output = row['gpt_output']

    try:
        # "" → " 치환 후 JSON 추출
        cleaned = gpt_output.replace('""', '"')
        json_match = re.search(r'\{[^{}]+\}', cleaned, re.DOTALL)

        if json_match:
            scores = json.loads(json_match.group())
            scores['participant_id'] = pid
            parsed_results.append(scores)
            print(f"[완료] {pid}번")
        else:
            print(f"[JSON 없음] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 데이터프레임 변환
df_parsed = pd.DataFrame(parsed_results)

# participant_id 맨 앞으로
cols = ['participant_id'] + [c for c in df_parsed.columns if c != 'participant_id']
df_parsed = df_parsed[cols]

df_parsed.to_csv(
    "/content/drive/MyDrive/26-1_URP/features/llm_features_deleted_parsed.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n저장 완료!")
print(f"참가자 수: {len(df_parsed)}")
print(df_parsed.head())

[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료!
참가자 수: 75
   participant_id  item2  item3  item6  item8  item9  item10  \
0               2      6      4      6      4      6       4   
1               3      5      5      5      5      5       5   
2               4      6      5      6      5      6       5   
3               5      5      6      5      6      5 

In [10]:
# 경로 설정
TRANSCRIPT_DIR = "/content/drive/MyDrive/26-1_URP/transcripts/"
OUTPUT_PATH = "/content/drive/MyDrive/26-1_URP/features/llm_features_deleted_yelin_confidence.csv"


# 프롬프트
system_prompt = """당신은 감정조절 전략을 평가하는 심리학 기반 채점자입니다.
아래 응답을 읽고 인지적 재평가와 억제 전략을 각각 1~7점 리커트 척도로 평가하세요.

인지적 재평가 점수:
1점 = 재평가가 전혀 없음. 상황의 의미를 다르게 해석하려는 시도가 없음.
2점 = 재평가 가능성이 매우 약함. 긍정적 표현은 있으나 구체적 관점 전환이 없음.
3점 = 막연한 긍정적 사고가 있음. 하지만 어떻게 의미를 바꿨는지 불분명함.
4점 = 상황을 다르게 보려는 시도가 어느 정도 있음. 다만 설명이 짧거나 구체성이 부족함.
5점 = 명확한 관점 전환 또는 의미 재해석이 있음.
6점 = 적극적인 의미 재구성이 있으며 감정 조절 의도가 뚜렷함.
7점 = 매우 명확하고 구체적인 의미 재구성이 있고, 그 결과 감정이 완화되거나 변화한 내용까지 나타남.

억제 점수:
1점 = 억제가 전혀 없음. 감정을 숨기거나 표현을 억누른 내용이 없음.
2점 = 억제 가능성이 매우 약함. 감정을 드러내지 않았을 가능성은 있으나 불명확함.
3점 = 감정을 참으려는 표현이 약하게 있음. 하지만 구체적 행동은 부족함.
4점 = 감정을 겉으로 드러내지 않으려는 시도가 어느 정도 나타남.
5점 = 감정 표현을 의도적으로 억제한 것이 명확함.
6점 = 강한 감정을 느꼈지만 표현하지 않으려는 의도와 행동이 뚜렷함.
7점 = 강한 감정을 철저히 숨기거나 아무렇지 않은 척하는 등 억제가 매우 명확함.

[세부 판단 기준]
인지적 재평가는 다음 요소가 많고 구체적일수록 높은 점수를 부여하세요:
- 상황을 다른 관점에서 해석함
- 타인의 입장이나 상황을 고려함
- 부정적 경험을 배움, 성장, 기회로 재구성함
- 사건의 원인이나 의미를 다르게 생각함
- 재해석 후 감정이 완화되거나 변화함

억제는 다음 요소가 많고 구체적일수록 높은 점수를 부여하세요:
- 부정적 감정을 느꼈음이 명확함
- 감정을 말, 표정, 행동으로 드러내지 않음
- 티 내지 않음, 참음, 숨김, 괜찮은 척함
- 일부러 감정 표현을 억누름
- 강한 감정을 느꼈지만 겉으로는 다른 모습을 보임

[구분 규칙]
- "다르게 생각했다", "이해하려 했다", "배움이라고 생각했다"처럼 상황의 의미를 바꾸면 인지적 재평가로 평가하세요.
- "참았다", "티 내지 않았다", "아무렇지 않은 척했다"처럼 감정 표현을 숨기면 억제로 평가하세요.
- 하나의 응답에 두 전략이 모두 있으면 둘 다 점수를 부여하고 main_strategy는 mixed로 표시하세요.
- 단순히 "생각했다"라는 단어만으로 재평가로 판단하지 마세요.
- 단순히 "조용히 있었다"만으로 높은 억제 점수를 주지 마세요. 감정을 숨기거나 표현을 억제한 맥락이 필요합니다.

[언어 유의사항]
전사본은 한국어로 작성되어 있습니다.
한국어 화자는 정서조절을 간접적으로 표현하는 경우가 많습니다.
"스스로를 다독였다", "별거 아니라고 생각했다", "그냥 넘기려 했다", "마음을 고쳐먹었다" 같은
간접적 표현도 각 전략의 근거로 볼 수 있습니다.

[출력 형식]
아래 세 단계를 순서대로 수행하세요.

Step 1 (Fact):
인지적 재평가와 억제 각각에 대해, 전사본에서 근거가 되는 표현을 직접 인용하세요.
근거가 없으면 반드시 "없음"이라고 명시하세요.

Step 2 (채점):
Step 1에서 인용한 근거를 바탕으로 각 전략의 점수를 부여하세요.
근거가 "없음"인 전략은 1~2점으로 제한하세요.

Step 3 (자기검증):
Step 2의 채점이 전사본 근거에 기반한 것인지, 아니면 추측인지 스스로 평가하세요.
"이 채점은 전사본에서 직접 확인한 근거에 기반하는가?"
- 그렇다 (True) → high 또는 medium
- 아니다 (False) → 반드시 low

그 다음 아래 JSON을 마지막에 출력하세요. JSON 이후에는 아무 텍스트도 추가하지 마세요:

{
  "cognitive_reappraisal": ,
  "expressive_suppression": ,
  "main_strategy": ,
  "confidence": ,
  "confidence_reason":
}
"""

# 실제 존재하는 참가자 번호 추출
pids = set()
for f in os.listdir(TRANSCRIPT_DIR):
    if f.endswith(".txt"):
        pid = int(f.split("_")[1])
        pids.add(pid)

print(f"총 참가자 수: {len(pids)}")

# 메인 루프
results = []

for pid in sorted(pids):
    txt_path = os.path.join(TRANSCRIPT_DIR, f"interview_{pid}_2.txt")  # 2번만 (stress 조건)

    if not os.path.exists(txt_path):
        print(f"[없음] {pid}번 전사본 없음")
        continue

    # 전사본 읽기
    with open(txt_path, "r", encoding="utf-8") as f:
        transcript = f.read()

    # GPT 호출
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"```{transcript}```"}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content
        results.append({
            "participant_id": pid,
            "gpt_output": result_text
        })
        print(f"[완료] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 저장
df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")
print(f"완료된 참가자 수: {len(df)}")

총 참가자 수: 75
[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료: /content/drive/MyDrive/26-1_URP/features/llm_features_deleted_yelin_confidence.csv
완료된 참가자 수: 75


In [12]:
import pandas as pd
import json
import re

# CSV 불러오기
df = pd.read_csv("/content/drive/MyDrive/26-1_URP/features/llm_features_deleted_yelin_confidence.csv")

parsed_results = []

for _, row in df.iterrows():
    pid = row['participant_id']
    gpt_output = row['gpt_output']

    try:
        # "" → " 치환 후 JSON 추출
        cleaned = gpt_output.replace('""', '"')
        json_match = re.search(r'\{[^{}]+\}', cleaned, re.DOTALL)

        if json_match:
            scores = json.loads(json_match.group())
            scores['participant_id'] = pid
            parsed_results.append(scores)
            print(f"[완료] {pid}번")
        else:
            print(f"[JSON 없음] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 데이터프레임 변환
df_parsed = pd.DataFrame(parsed_results)

# participant_id 맨 앞으로
cols = ['participant_id'] + [c for c in df_parsed.columns if c != 'participant_id']
df_parsed = df_parsed[cols]

df_parsed.to_csv(
    "/content/drive/MyDrive/26-1_URP/features/llm_features_deleted_yelin_confidence_parsed.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n저장 완료!")
print(f"참가자 수: {len(df_parsed)}")
print(df_parsed.head())

[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료!
참가자 수: 75
   participant_id  cognitive_reappraisal  expressive_suppression  \
0               2                      5                       6   
1               3                      4                       5   
2               4                      5                       2   
3               5                    

In [11]:
# 경로 설정
TRANSCRIPT_DIR = "/content/drive/MyDrive/26-1_URP/transcripts/"
OUTPUT_PATH = "/content/drive/MyDrive/26-1_URP/features/llm_features_deleted_confidence.csv"

system_prompt = """
역할
당신은 심리학자이자 정서조절 연구자, 정신건강 전문가입니다.

지시사항
아래 세 개의 백틱(```) 안에 있는 텍스트는 면접자와 참가자 간의 인터뷰 전사본입니다.
이 인터뷰는 참가자가 경험한 스트레스 상황 또는 정서적으로 중요한 사건에 관한 것입니다.
아래의 정서조절 전략 척도(ERQ)에 해당하는 경향을 참가자가 얼마나 보이는지 평가해주세요.
ERQ는 7점 척도로 정서조절 경향을 측정합니다:

"나는 감정을 혼자 속으로 삭인다."
"나는 덜 부정적인 감정(예: 슬픔, 분노)을 느끼고 싶을 때, 생각하는 내용을 바꾼다."
"나는 감정을 표현하지 않음으로써 감정을 통제한다."
"나는 내가 처한 상황을 생각하는 방식을 바꿈으로써 감정을 통제한다."
"나는 부정적인 감정을 느낄 때, 그것을 표현하지 않으려고 노력한다."
"나는 덜 부정적인 감정을 느끼고 싶을 때, 상황을 바라보는 방식을 바꾼다."

인지적 재평가(Cognitive Reappraisal)는 상황의 해석을 바꿔 정서적 영향을 변화시키는 전략입니다.
표현 억제(Expressive Suppression)는 감정 표현을 억제하는 전략입니다.

참가자의 응답은 아래 척도로 평가합니다:
1: 전혀 그렇지 않다
2: 그렇지 않다
3: 약간 그렇지 않다
4: 보통이다
5: 약간 그렇다
6: 그렇다
7: 매우 그렇다

언어 유의사항
전사본은 한국어로 작성되어 있습니다.
한국어 화자는 정서조절을 간접적으로 표현하는 경우가 많습니다.
예를 들어 "스스로를 다독였다", "그냥 넘기려 했다", "마음을 고쳐먹었다", "별거 아니라고 생각했다" 같은 표현도 인지적 재평가나 표현 억제의 증거로 볼 수 있습니다.
명시적인 표현이 없더라도 맥락상 강하게 암시되는 경우 점수에 반영하세요.

평가 절차
각 항목에 대해 아래 세 단계를 수행하세요.

Step 1 (Fact):
각 항목별로 전사본에서 근거가 되는 표현을 직접 인용하세요.
근거가 없으면 반드시 "없음"이라고 명시하세요.

Step 2 (채점):
Step 1에서 인용한 근거를 바탕으로 1~7점을 부여하세요.
근거가 "없음"인 항목은 1~2점으로 제한하세요.
전사본에서 명시적으로 드러나거나 강하게 암시된 근거에만 기반하여 점수를 매기세요.
정서적 경험과 의도적인 정서조절을 구분하세요.

Step 3 (자기검증):
Step 2의 채점이 전사본 근거에 기반한 것인지 스스로 평가하세요.
"이 채점은 전사본에서 직접 확인한 근거에 기반하는가?"
- 그렇다 (True) → confidence: high 또는 medium
- 아니다 (False) → confidence: 반드시 low
항목 전체를 종합하여 confidence를 하나로 결정하세요.

항목별 채점 기준
1, 4, 7은 기준점일 뿐이며, 1~7 중 어느 점수든 선택할 수 있습니다.

항목 2
1 = 감정을 공개적으로 표현함
4 = 감정을 숨기려는 중간 정도의 경향
7 = 타인에게 감정을 명시적이고 일관되게 숨김

항목 3
1 = 부정 감정을 줄이기 위한 사고 전환 시도 없음
4 = 부정적 사고를 전환하려는 일부 시도
7 = 부정 감정을 줄인 명시적이고 지속적인 사고 전환 과정

항목 6
1 = 표현을 통해 감정을 조절함 (억제 없음)
4 = 비표현에 중간 정도 의존
7 = 감정을 표현하지 않음으로써 통제해야 한다는 명시적 신념

항목 8
1 = 인지적 정서조절의 증거 없음
4 = 중간 정도의 재해석 경향
7 = 인지적 재구성을 주요 정서 통제 전략으로 명시적으로 사용

항목 9
1 = 부정 감정을 공개적으로 표현함
4 = 부정 감정을 숨기려는 중간 정도의 노력
7 = 부정적 감정 표현을 명시적이고 지속적으로 억제함

항목 10
1 = 재해석 노력 없음
4 = 스트레스 상황에 대한 일부 재해석
7 = 정서적 회복이나 안도감을 가져온 상세한 재해석

최종 목표
ERQ 각 항목에 대해 1~7점을 부여하세요.
추가로 아래를 계산하세요:
인지적 재평가 평균 점수 (항목 3, 8, 10의 평균)
표현 억제 평균 점수 (항목 2, 6, 9의 평균)

출력 형식
각 항목에 대해 Step 1~3을 수행한 뒤,
아래 JSON을 마지막에 출력하세요. JSON 이후에는 아무 텍스트도 추가하지 마세요:

{
  "item2": ,
  "item3": ,
  "item6": ,
  "item8": ,
  "item9": ,
  "item10": ,
  "cognitive_reappraisal_mean": ,
  "expressive_suppression_mean": ,
  "confidence": ,
  "confidence_reason":
}
"""

# 실제 존재하는 참가자 번호 추출
pids = set()
for f in os.listdir(TRANSCRIPT_DIR):
    if f.endswith(".txt"):
        pid = int(f.split("_")[1])
        pids.add(pid)

print(f"총 참가자 수: {len(pids)}")

# 메인 루프
results = []

for pid in sorted(pids):
    txt_path = os.path.join(TRANSCRIPT_DIR, f"interview_{pid}_2.txt")  # 2번만 (stress 조건)

    if not os.path.exists(txt_path):
        print(f"[없음] {pid}번 전사본 없음")
        continue

    # 전사본 읽기
    with open(txt_path, "r", encoding="utf-8") as f:
        transcript = f.read()

    # GPT 호출
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"```{transcript}```"}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content
        results.append({
            "participant_id": pid,
            "gpt_output": result_text
        })
        print(f"[완료] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 저장
df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")
print(f"완료된 참가자 수: {len(df)}")

총 참가자 수: 75
[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료: /content/drive/MyDrive/26-1_URP/features/llm_features_deleted_confidence.csv
완료된 참가자 수: 75


In [13]:
import pandas as pd
import json
import re

# CSV 불러오기
df = pd.read_csv("/content/drive/MyDrive/26-1_URP/features/llm_features_deleted_confidence.csv")  # 파일명 확인

parsed_results = []

for _, row in df.iterrows():
    pid = row['participant_id']
    gpt_output = row['gpt_output']

    try:
        cleaned = gpt_output.replace('""', '"')
        json_match = re.search(r'\{[^{}]+\}', cleaned, re.DOTALL)

        if json_match:
            scores = json.loads(json_match.group())
            scores['participant_id'] = pid
            parsed_results.append(scores)
            print(f"[완료] {pid}번")
        else:
            print(f"[JSON 없음] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

df_parsed = pd.DataFrame(parsed_results)
cols = ['participant_id'] + [c for c in df_parsed.columns if c != 'participant_id']
df_parsed = df_parsed[cols]

df_parsed.to_csv(
    "/content/drive/MyDrive/26-1_URP/features/llm_features_deleted_confidence_parsed.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n저장 완료!")
print(f"참가자 수: {len(df_parsed)}")
print(df_parsed.head())

[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료!
참가자 수: 75
   participant_id  item2  item3  item6  item8  item9  item10  \
0               2      6      4      6      4      6       4   
1               3      4      5      4      5      4       5   
2               4      3      5      3      5      3       5   
3               5      4      5      5      6      4 

In [15]:
# 경로 설정
TRANSCRIPT_DIR = "/content/drive/MyDrive/26-1_URP/transcripts/"
OUTPUT_PATH = "/content/drive/MyDrive/26-1_URP/features/llm_features_deleted_confidence_reverse.csv"

system_prompt = """
역할
당신은 심리학자이자 정서조절 연구자, 정신건강 전문가입니다.

지시사항
아래 세 개의 백틱(```) 안에 있는 텍스트는 면접자와 참가자 간의 인터뷰 전사본입니다.
이 인터뷰는 참가자가 경험한 스트레스 상황 또는 정서적으로 중요한 사건에 관한 것입니다.
아래의 정서조절 전략 척도(ERQ)에 해당하는 경향을 참가자가 얼마나 보이는지 평가해주세요.
ERQ는 7점 척도로 정서조절 경향을 측정합니다:

"나는 감정을 혼자 속으로 삭인다."
"나는 덜 부정적인 감정(예: 슬픔, 분노)을 느끼고 싶을 때, 생각하는 내용을 바꾼다."
"나는 감정을 표현하지 않음으로써 감정을 통제한다."
"나는 내가 처한 상황을 생각하는 방식을 바꿈으로써 감정을 통제한다."
"나는 부정적인 감정을 느낄 때, 그것을 표현하지 않으려고 노력한다."
"나는 덜 부정적인 감정을 느끼고 싶을 때, 상황을 바라보는 방식을 바꾼다."

인지적 재평가(Cognitive Reappraisal)는 상황의 해석을 바꿔 정서적 영향을 변화시키는 전략입니다.
표현 억제(Expressive Suppression)는 감정 표현을 억제하는 전략입니다.

참가자의 응답은 아래 척도로 평가합니다:
1: 전혀 그렇지 않다
2: 그렇지 않다
3: 약간 그렇지 않다
4: 보통이다
5: 약간 그렇다
6: 그렇다
7: 매우 그렇다

언어 유의사항
전사본은 한국어로 작성되어 있습니다.
한국어 화자는 정서조절을 간접적으로 표현하는 경우가 많습니다.
예를 들어 "스스로를 다독였다", "그냥 넘기려 했다", "마음을 고쳐먹었다", "별거 아니라고 생각했다" 같은 표현도 인지적 재평가나 표현 억제의 증거로 볼 수 있습니다.
명시적인 표현이 없더라도 맥락상 강하게 암시되는 경우 점수에 반영하세요.

평가 절차
각 항목에 대해 아래 세 단계를 수행하세요.

Step 1 (Fact):
각 항목별로 전사본에서 근거가 되는 표현을 직접 인용하세요.
근거가 없으면 반드시 "없음"이라고 명시하세요.

Step 2 (채점):
Step 1에서 인용한 근거를 바탕으로 1~7점을 부여하세요.
근거가 "없음"인 항목은 1~2점으로 제한하세요.
전사본에서 명시적으로 드러나거나 강하게 암시된 근거에만 기반하여 점수를 매기세요.
정서적 경험과 의도적인 정서조절을 구분하세요.

Step 3 (자기검증):
Step 2의 채점이 전사본 근거에 기반한 것인지 스스로 평가하세요.
"이 채점은 전사본에서 직접 확인한 근거에 기반하는가?"
- 그렇다 (True) → confidence: high 또는 medium
- 아니다 (False) → confidence: 반드시 low
항목 전체를 종합하여 confidence를 하나로 결정하세요.

항목별 채점 기준
1, 4, 7은 기준점일 뿐이며, 1~7 중 어느 점수든 선택할 수 있습니다.

항목 2
1 = 감정을 공개적으로 표현함
4 = 감정을 숨기려는 중간 정도의 경향
7 = 타인에게 감정을 명시적이고 일관되게 숨김

항목 3
1 = 부정 감정을 줄이기 위한 사고 전환 시도 없음
4 = 부정적 사고를 전환하려는 일부 시도
7 = 부정 감정을 줄인 명시적이고 지속적인 사고 전환 과정

항목 6
1 = 표현을 통해 감정을 조절함 (억제 없음)
4 = 비표현에 중간 정도 의존
7 = 감정을 표현하지 않음으로써 통제해야 한다는 명시적 신념

항목 8
1 = 인지적 정서조절의 증거 없음
4 = 중간 정도의 재해석 경향
7 = 인지적 재구성을 주요 정서 통제 전략으로 명시적으로 사용

항목 9
1 = 부정 감정을 공개적으로 표현함
4 = 부정 감정을 숨기려는 중간 정도의 노력
7 = 부정적 감정 표현을 명시적이고 지속적으로 억제함

항목 10
1 = 재해석 노력 없음
4 = 스트레스 상황에 대한 일부 재해석
7 = 정서적 회복이나 안도감을 가져온 상세한 재해석

최종 목표
ERQ 각 항목에 대해 1~7점을 부여하세요.
추가로 아래를 계산하세요:
인지적 재평가 평균 점수 (항목 3, 8, 10의 평균)
표현 억제 평균 점수 (항목 2, 6, 9의 평균)

출력 형식
각 항목에 대해 Step 1~3을 수행한 뒤,
아래 JSON을 마지막에 출력하세요. JSON 이후에는 아무 텍스트도 추가하지 마세요:

{
  "confidence": ,
  "confidence_reason":,
  "item2": ,
  "item3": ,
  "item6": ,
  "item8": ,
  "item9": ,
  "item10": ,
  "cognitive_reappraisal_mean": ,
  "expressive_suppression_mean":
}
"""

# 실제 존재하는 참가자 번호 추출
pids = set()
for f in os.listdir(TRANSCRIPT_DIR):
    if f.endswith(".txt"):
        pid = int(f.split("_")[1])
        pids.add(pid)

print(f"총 참가자 수: {len(pids)}")

# 메인 루프
results = []

for pid in sorted(pids):
    txt_path = os.path.join(TRANSCRIPT_DIR, f"interview_{pid}_2.txt")  # 2번만 (stress 조건)

    if not os.path.exists(txt_path):
        print(f"[없음] {pid}번 전사본 없음")
        continue

    # 전사본 읽기
    with open(txt_path, "r", encoding="utf-8") as f:
        transcript = f.read()

    # GPT 호출
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"```{transcript}```"}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content
        results.append({
            "participant_id": pid,
            "gpt_output": result_text
        })
        print(f"[완료] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 저장
df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")
print(f"완료된 참가자 수: {len(df)}")

총 참가자 수: 75
[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료: /content/drive/MyDrive/26-1_URP/features/llm_features_deleted_confidence_reverse.csv
완료된 참가자 수: 75


In [23]:
# 경로 설정
TRANSCRIPT_DIR = "/content/drive/MyDrive/26-1_URP/transcripts/"
OUTPUT_PATH = "/content/drive/MyDrive/26-1_URP/features/llm_features_yelin2.csv"

system_prompt = """
당신은 감정조절 전략을 평가하는 심리학 기반 채점자입니다.

아래 응답을 읽고, 감정조절 전략 중
1) 인지적 재평가(cognitive reappraisal)
2) 억제(expressive suppression)

두 전략을 각각 1~7점 리커트 척도로 평가하세요.

평가의 핵심은 “전략이 얼마나 명확하고 구체적으로 드러나는가”입니다.
애매하거나 추론에 의존해야 하는 경우에는 중간 점수로 처리하지 말고, 근거가 부족하면 낮은 점수를 부여하세요.

특히 4점은 애매할 때 주는 기본값이 아닙니다.
전략의 명확한 근거가 부족하면 1~3점을 적극적으로 사용하고,
전략의 과정과 결과가 구체적으로 드러나면 6~7점을 적극적으로 사용하세요.

==================================================
[평가 대상 응답]
{response}
==================================================


# 1. 인지적 재평가 점수 기준

인지적 재평가는 부정적이거나 감정적인 상황의 의미를 다르게 해석하여 감정을 조절하려는 전략입니다.

다음 요소가 많고 구체적일수록 높은 점수를 부여하세요.

- 상황을 다른 관점에서 해석함
- 타인의 입장이나 상황을 고려함
- 부정적 경험을 배움, 성장, 기회로 재구성함
- 사건의 원인이나 의미를 다르게 생각함
- 재해석 후 감정이 완화되거나 변화함


## 1점 = 재평가 없음
상황의 의미를 바꾸려는 시도가 전혀 없음.
단순한 감정 표현, 행동 묘사, 회피, 체념만 있음.

예:
“그냥 너무 화가 났다.”
“아무 말도 하지 않았다.”
“기분이 나빴다.”


## 2점 = 매우 약한 재평가 가능성
긍정적인 말이나 위로 표현은 있지만, 상황을 실제로 다르게 해석한 내용은 거의 없음.
채점자가 추론해야만 재평가로 볼 수 있음.

예:
“괜찮아질 거라고 생각했다.”
“좋게 생각하려고 했다.”


## 3점 = 막연한 긍정적 사고
긍정적으로 생각하려는 시도는 있으나, 무엇을 어떻게 다르게 해석했는지 불명확함.
구체적인 의미 변화가 부족함.

예:
“너무 부정적으로 생각하지 않으려고 했다.”
“좋은 경험이라고 생각하려 했다.”


## 4점 = 약한 관점 전환
상황을 다르게 보려는 시도가 어느 정도 있으나, 설명이 짧고 구체성이 제한적임.
타인의 입장, 배움, 성장, 원인 재해석 중 하나가 간단히 나타남.

예:
“상대방도 사정이 있었을 거라고 생각했다.”
“이번 일을 통해 배울 점이 있다고 생각했다.”


## 5점 = 명확한 재평가
상황의 의미를 다르게 해석한 내용이 분명함.
부정적 사건을 다른 관점, 배움, 기회, 상대의 상황 등으로 재구성함.

예:
“처음에는 무시당한 것 같아 속상했지만, 상대가 바빠서 그랬을 수도 있다고 생각했다.”
“실패라고만 생각하지 않고 내가 부족한 부분을 확인할 기회라고 봤다.”


## 6점 = 적극적인 의미 재구성
상황의 의미를 능동적으로 바꾸고, 감정을 조절하려는 의도가 뚜렷함.
구체적인 근거와 함께 해석이 변화함.

예:
“처음에는 비난처럼 느껴졌지만, 나를 공격하려는 말이 아니라 개선점을 알려주는 피드백이라고 생각하니 덜 속상했다.”


## 7점 = 매우 강하고 구체적인 재평가
초기 감정, 재해석 과정, 새로운 의미, 감정 변화가 모두 뚜렷함.
부정적 경험을 구체적으로 성장, 배움, 관계 이해, 기회로 재구성하고 그 결과 감정이 완화됨.

예:
“처음에는 내가 무시당했다고 느껴 화가 났지만, 상대가 최근 힘든 상황에 있었고 내 말을 공격하려는 의도는 아니었을 수 있다고 생각했다. 그렇게 보니 화가 줄었고, 이 일을 대화를 통해 서로를 더 이해할 기회로 받아들이게 되었다.”


# 2. 억제 점수 기준

억제는 느낀 감정을 겉으로 드러내지 않거나, 말·표정·행동으로 표현하지 않으려는 전략입니다.

다음 요소가 많고 구체적일수록 높은 점수를 부여하세요.

- 부정적 감정을 느꼈음이 명확함
- 감정을 말, 표정, 행동으로 드러내지 않음
- 티 내지 않음, 참음, 숨김, 괜찮은 척함
- 일부러 감정 표현을 억누름
- 강한 감정을 느꼈지만 겉으로는 다른 모습을 보임


## 1점 = 억제 없음
감정을 숨기거나 참았다는 내용이 전혀 없음.
감정을 그대로 표현했거나, 단순히 감정만 묘사함.

예:
“화가 나서 바로 말했다.”
“너무 속상했다.”


## 2점 = 억제 가능성 매우 약함
감정을 드러내지 않았을 가능성은 있지만, 명확한 억제 표현은 없음.
“조용히 있었다”처럼 감정 억제인지 단순 행동인지 불분명함.

예:
“그냥 가만히 있었다.”
“말을 많이 하지 않았다.”


## 3점 = 약한 억제
감정을 참으려는 표현이 약하게 나타남.
다만 어떤 감정을 어떻게 숨겼는지는 구체적이지 않음.

예:
“조금 참았다.”
“화가 났지만 말은 아꼈다.”


## 4점 = 어느 정도의 억제
부정적 감정을 느꼈고, 이를 겉으로 드러내지 않으려는 시도가 나타남.
하지만 억제의 강도나 구체적 행동은 제한적임.

예:
“속상했지만 티를 내지 않으려고 했다.”
“기분이 나빴지만 분위기를 망치고 싶지 않아 말하지 않았다.”


## 5점 = 명확한 억제
감정을 의도적으로 숨기거나 표현하지 않은 것이 분명함.
말, 표정, 행동 중 하나 이상에서 억제가 나타남.

예:
“화가 많이 났지만 표정에 드러내지 않으려고 웃으면서 넘겼다.”
“속상했지만 괜찮은 척하며 아무 말도 하지 않았다.”


## 6점 = 강한 억제
강한 감정을 느꼈지만 이를 드러내지 않으려는 의도와 행동이 모두 뚜렷함.
감정을 억누르기 위해 의식적으로 표정, 말투, 행동을 조절함.

예:
“정말 울고 싶을 만큼 속상했지만, 사람들이 걱정할까 봐 일부러 웃고 평소처럼 행동했다.”


## 7점 = 매우 강하고 철저한 억제
강한 감정을 거의 완전히 숨김.
속마음과 겉으로 보인 모습의 차이가 매우 뚜렷하며, 아무렇지 않은 척하거나 평소처럼 행동한 내용이 구체적으로 나타남.

예:
“속으로는 너무 화가 나고 눈물이 날 것 같았지만, 절대 티 내지 않으려고 표정과 목소리를 평소처럼 유지했고, 오히려 괜찮다고 말하며 끝까지 아무렇지 않은 척했다.”


# 3. 구분 규칙

다음 규칙을 반드시 따르세요.

- “다르게 생각했다”, “이해하려 했다”, “배움이라고 생각했다”처럼 상황의 의미를 바꾸면 인지적 재평가로 평가하세요.
- “참았다”, “티 내지 않았다”, “아무렇지 않은 척했다”처럼 감정 표현을 숨기면 억제로 평가하세요.
- 하나의 응답에 두 전략이 모두 있으면 둘 다 점수를 부여하고 main_strategy는 mixed로 표시하세요.
- 단순히 “생각했다”라는 단어만으로 재평가로 판단하지 마세요.
- 단순히 “조용히 있었다”만으로 높은 억제 점수를 주지 마세요. 감정을 숨기거나 표현을 억제한 맥락이 필요합니다.
- “좋게 생각했다”, “긍정적으로 생각했다”처럼 구체적인 의미 변화가 없는 표현은 인지적 재평가 2~3점에 머물러야 합니다.
- “참았다”만 있고 어떤 감정을 어떻게 숨겼는지 불분명하면 억제 3점 이하로 평가하세요.
- 5점 이상은 반드시 명확한 전략 사용 근거가 있을 때만 부여하세요.
- 6~7점은 전략 사용의 의도, 구체적 과정, 감정 변화 또는 표현 억제 행동이 함께 나타날 때 부여하세요.


# 4. 점수 분산을 위한 판단 원칙

채점 시 4~6점에 점수가 몰리지 않도록 다음 원칙을 적용하세요.

## 낮은 점수 1~3점을 적극적으로 사용해야 하는 경우

- 전략이 명확하지 않고 추론에 의존해야 하는 경우
- 단순 감정 표현만 있는 경우
- 단순 행동 묘사만 있는 경우
- 표현은 있으나 구체성이 거의 없는 경우
- 감정 조절 전략보다 단순 회피, 무기력, 체념에 가까운 경우
- 응답이 짧아서 전략 사용 근거가 부족한 경우

예:
“그냥 시간이 지나면 괜찮아질 거라고 생각했다.”
→ 인지적 재평가 2~3점

“말하지 않고 조용히 있었다.”
→ 억제 2점

단, “화가 났지만 티 내지 않으려고 조용히 있었다”라면 억제 4점 이상 가능


## 높은 점수 6~7점을 적극적으로 사용해야 하는 경우

- 초기 부정 감정이 명확함
- 전략 사용 의도가 분명함
- 구체적인 생각 변화 또는 행동 조절이 나타남
- 재평가의 경우 감정 변화가 드러남
- 억제의 경우 속마음과 겉모습의 차이가 뚜렷함
- 전략 사용 전후의 변화가 나타남

예:
“처음에는 실패라고 느껴 좌절했지만, 내가 무엇을 보완해야 하는지 알게 된 기회라고 생각하니 마음이 조금 편해졌다.”
→ 인지적 재평가 7점

“너무 화가 났지만 회의 분위기를 해치고 싶지 않아서 표정과 말투를 최대한 평소처럼 유지했다.”
→ 억제 6~7점


# 5. main_strategy 분류 기준

응답에서 더 핵심적으로 사용된 전략을 기준으로 main_strategy를 표시하세요.

- reappraisal: 인지적 재평가가 주된 전략
- suppression: 억제가 주된 전략
- mixed: 두 전략이 모두 명확하게 나타남
- none: 두 전략 모두 뚜렷하지 않음

mixed는 상황을 다르게 해석하는 내용과 감정을 숨기는 내용이 모두 명확하게 있을 때 사용하세요.

예:
“처음에는 화가 났지만 상대도 힘든 상황이었을 거라고 생각했고, 다른 사람들 앞에서는 티 내지 않으려고 웃었다.”
→ main_strategy: mixed


# 6. confidence 평가 기준

채점 확실성을 high, medium, low 중 하나로 표시하세요.

## high
전략 사용이 명확하고 점수 판단 근거가 충분함.

## medium
전략이 어느 정도 보이지만 구체성이 부족하거나 점수 경계에 있음.

## low
전략 여부가 매우 애매하거나, 단어만 있고 실제 맥락이 부족함.
연구자 재검토가 필요함.

low confidence는 다음과 같은 경우에 부여하세요.

- 응답이 지나치게 짧음
- “생각했다”, “참았다”, “넘겼다”, “조용히 있었다”처럼 단어만 있고 맥락이 부족함
- 재평가인지 단순 긍정인지 구분이 어려움
- 억제인지 단순 침묵인지 구분이 어려움
- 두 전략 점수 모두 2~4점 사이에서 애매하게 판단됨


# 7. 출력 형식

반드시 아래 JSON 형식으로만 출력하세요.
설명 문장이나 추가 코멘트는 JSON 밖에 쓰지 마세요.

{
"reappraisal_score": 1,
"suppression_score": 1,
"main_strategy": "reappraisal | suppression | mixed | none",
"confidence": "high | medium | low",
"rationale": "점수 판단 근거를 1~2문장으로 간단히 설명"
}
"""

# 실제 존재하는 참가자 번호 추출
pids = set()
for f in os.listdir(TRANSCRIPT_DIR):
    if f.endswith(".txt"):
        pid = int(f.split("_")[1])
        pids.add(pid)

print(f"총 참가자 수: {len(pids)}")

# 메인 루프
results = []

for pid in sorted(pids):
    txt_path = os.path.join(TRANSCRIPT_DIR, f"interview_{pid}_2.txt")  # 2번만 (stress 조건)

    if not os.path.exists(txt_path):
        print(f"[없음] {pid}번 전사본 없음")
        continue

    # 전사본 읽기
    with open(txt_path, "r", encoding="utf-8") as f:
        transcript = f.read()

    # GPT 호출
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"```{transcript}```"}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content
        results.append({
            "participant_id": pid,
            "gpt_output": result_text
        })
        print(f"[완료] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 저장
df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")
print(f"완료된 참가자 수: {len(df)}")


총 참가자 수: 75
[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료: /content/drive/MyDrive/26-1_URP/features/llm_features_yelin2.csv
완료된 참가자 수: 75


In [24]:

# CSV 불러오기
df = pd.read_csv("/content/drive/MyDrive/26-1_URP/features/llm_features_yelin2.csv")  # 파일명 확인

parsed_results = []

for _, row in df.iterrows():
    pid = row['participant_id']
    gpt_output = row['gpt_output']

    try:
        cleaned = gpt_output.replace('""', '"')
        json_match = re.search(r'\{[^{}]+\}', cleaned, re.DOTALL)

        if json_match:
            scores = json.loads(json_match.group())
            scores['participant_id'] = pid
            parsed_results.append(scores)
            print(f"[완료] {pid}번")
        else:
            print(f"[JSON 없음] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

df_parsed = pd.DataFrame(parsed_results)
cols = ['participant_id'] + [c for c in df_parsed.columns if c != 'participant_id']
df_parsed = df_parsed[cols]

df_parsed.to_csv(
    "/content/drive/MyDrive/26-1_URP/features/llm_features_yelin2_parsed.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n저장 완료!")
print(f"참가자 수: {len(df_parsed)}")
print(df_parsed.head())

[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료!
참가자 수: 75
   participant_id  reappraisal_score  suppression_score main_strategy  \
0               2                  4                  6         mixed   
1               3                  2                  4         mixed   
2               4                  5                  4         mixed   
3               5

In [25]:
# 경로 설정
TRANSCRIPT_DIR = "/content/drive/MyDrive/26-1_URP/transcripts/"
OUTPUT_PATH = "/content/drive/MyDrive/26-1_URP/features/llm_features_lang_del.csv"

system_prompt = """
역할
당신은 심리학자이자 정서조절 연구자, 정신건강 전문가입니다.

지시사항
아래 세 개의 백틱(```) 안에 있는 텍스트는 면접자와 참가자 간의 인터뷰 전사본입니다.
이 인터뷰는 참가자가 경험한 스트레스 상황 또는 정서적으로 중요한 사건에 관한 것입니다.
아래의 정서조절 전략 척도(ERQ)에 해당하는 경향을 참가자가 얼마나 보이는지 평가해주세요.
ERQ는 7점 척도로 정서조절 경향을 측정합니다:

"나는 감정을 혼자 속으로 삭인다."
"나는 덜 부정적인 감정(예: 슬픔, 분노)을 느끼고 싶을 때, 생각하는 내용을 바꾼다."
"나는 감정을 표현하지 않음으로써 감정을 통제한다."
"나는 내가 처한 상황을 생각하는 방식을 바꿈으로써 감정을 통제한다."
"나는 부정적인 감정을 느낄 때, 그것을 표현하지 않으려고 노력한다."
"나는 덜 부정적인 감정을 느끼고 싶을 때, 상황을 바라보는 방식을 바꾼다."

인지적 재평가(Cognitive Reappraisal)는 상황의 해석을 바꿔 정서적 영향을 변화시키는 전략입니다.
표현 억제(Expressive Suppression)는 감정 표현을 억제하는 전략입니다.

참가자의 응답은 아래 척도로 평가합니다:
1: 전혀 그렇지 않다
2: 그렇지 않다
3: 약간 그렇지 않다
4: 보통이다
5: 약간 그렇다
6: 그렇다
7: 매우 그렇다

평가 절차
각 항목에 대해 아래 세 단계를 수행하세요.

Step 1 (Fact):
각 항목별로 전사본에서 근거가 되는 표현을 직접 인용하세요.
근거가 없으면 반드시 "없음"이라고 명시하세요.

Step 2 (채점):
Step 1에서 인용한 근거를 바탕으로 1~7점을 부여하세요.
근거가 "없음"인 항목은 1~2점으로 제한하세요.
전사본에서 명시적으로 드러나거나 강하게 암시된 근거에만 기반하여 점수를 매기세요.
정서적 경험과 의도적인 정서조절을 구분하세요.

Step 3 (자기검증):
Step 2의 채점이 전사본 근거에 기반한 것인지 스스로 평가하세요.
"이 채점은 전사본에서 직접 확인한 근거에 기반하는가?"
- 그렇다 (True) → confidence: high 또는 medium
- 아니다 (False) → confidence: 반드시 low
항목 전체를 종합하여 confidence를 하나로 결정하세요.

항목별 채점 기준
1, 4, 7은 기준점일 뿐이며, 1~7 중 어느 점수든 선택할 수 있습니다.

항목 2
1 = 감정을 공개적으로 표현함
4 = 감정을 숨기려는 중간 정도의 경향
7 = 타인에게 감정을 명시적이고 일관되게 숨김

항목 3
1 = 부정 감정을 줄이기 위한 사고 전환 시도 없음
4 = 부정적 사고를 전환하려는 일부 시도
7 = 부정 감정을 줄인 명시적이고 지속적인 사고 전환 과정

항목 6
1 = 표현을 통해 감정을 조절함 (억제 없음)
4 = 비표현에 중간 정도 의존
7 = 감정을 표현하지 않음으로써 통제해야 한다는 명시적 신념

항목 8
1 = 인지적 정서조절의 증거 없음
4 = 중간 정도의 재해석 경향
7 = 인지적 재구성을 주요 정서 통제 전략으로 명시적으로 사용

항목 9
1 = 부정 감정을 공개적으로 표현함
4 = 부정 감정을 숨기려는 중간 정도의 노력
7 = 부정적 감정 표현을 명시적이고 지속적으로 억제함

항목 10
1 = 재해석 노력 없음
4 = 스트레스 상황에 대한 일부 재해석
7 = 정서적 회복이나 안도감을 가져온 상세한 재해석

최종 목표
ERQ 각 항목에 대해 1~7점을 부여하세요.
추가로 아래를 계산하세요:
인지적 재평가 평균 점수 (항목 3, 8, 10의 평균)
표현 억제 평균 점수 (항목 2, 6, 9의 평균)

출력 형식
각 항목에 대해 Step 1~3을 수행한 뒤,
아래 JSON을 마지막에 출력하세요. JSON 이후에는 아무 텍스트도 추가하지 마세요:

{
  "confidence": ,
  "confidence_reason":,
  "item2": ,
  "item3": ,
  "item6": ,
  "item8": ,
  "item9": ,
  "item10": ,
  "cognitive_reappraisal_mean": ,
  "expressive_suppression_mean":
}
"""

# 실제 존재하는 참가자 번호 추출
pids = set()
for f in os.listdir(TRANSCRIPT_DIR):
    if f.endswith(".txt"):
        pid = int(f.split("_")[1])
        pids.add(pid)

print(f"총 참가자 수: {len(pids)}")

# 메인 루프
results = []

for pid in sorted(pids):
    txt_path = os.path.join(TRANSCRIPT_DIR, f"interview_{pid}_2.txt")  # 2번만 (stress 조건)

    if not os.path.exists(txt_path):
        print(f"[없음] {pid}번 전사본 없음")
        continue

    # 전사본 읽기
    with open(txt_path, "r", encoding="utf-8") as f:
        transcript = f.read()

    # GPT 호출
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"```{transcript}```"}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content
        results.append({
            "participant_id": pid,
            "gpt_output": result_text
        })
        print(f"[완료] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 저장
df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")
print(f"완료된 참가자 수: {len(df)}")

총 참가자 수: 75
[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료: /content/drive/MyDrive/26-1_URP/features/llm_features_lang_del.csv
완료된 참가자 수: 75


In [26]:
# CSV 불러오기
df = pd.read_csv("/content/drive/MyDrive/26-1_URP/features/llm_features_lang_del.csv")  # 파일명 확인

parsed_results = []

for _, row in df.iterrows():
    pid = row['participant_id']
    gpt_output = row['gpt_output']

    try:
        cleaned = gpt_output.replace('""', '"')
        json_match = re.search(r'\{[^{}]+\}', cleaned, re.DOTALL)

        if json_match:
            scores = json.loads(json_match.group())
            scores['participant_id'] = pid
            parsed_results.append(scores)
            print(f"[완료] {pid}번")
        else:
            print(f"[JSON 없음] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

df_parsed = pd.DataFrame(parsed_results)
cols = ['participant_id'] + [c for c in df_parsed.columns if c != 'participant_id']
df_parsed = df_parsed[cols]

df_parsed.to_csv(
    "/content/drive/MyDrive/26-1_URP/features/llm_features_lang_del_parsed.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n저장 완료!")
print(f"참가자 수: {len(df_parsed)}")
print(df_parsed.head())

[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료!
참가자 수: 75
   participant_id confidence  \
0               2       high   
1               3     medium   
2               4       high   
3               5       high   
4               6       high   

                                   confidence_reason  item2  item3  item6  \
0  모든 항목에서 참가자의 발언이 명확하게 정서조절 전략을 반영하고 

In [27]:
# 경로 설정
TRANSCRIPT_DIR = "/content/drive/MyDrive/26-1_URP/transcripts/"
OUTPUT_PATH = "/content/drive/MyDrive/26-1_URP/features/llm_features_deleted_yelin_lang_del.csv"


# 프롬프트
system_prompt = """당신은 감정조절 전략을 평가하는 심리학 기반 채점자입니다.
아래 응답을 읽고 인지적 재평가와 억제 전략을 각각 1~7점 리커트 척도로 평가하세요.

인지적 재평가 점수:
1점 = 재평가가 전혀 없음. 상황의 의미를 다르게 해석하려는 시도가 없음.
2점 = 재평가 가능성이 매우 약함. 긍정적 표현은 있으나 구체적 관점 전환이 없음.
3점 = 막연한 긍정적 사고가 있음. 하지만 어떻게 의미를 바꿨는지 불분명함.
4점 = 상황을 다르게 보려는 시도가 어느 정도 있음. 다만 설명이 짧거나 구체성이 부족함.
5점 = 명확한 관점 전환 또는 의미 재해석이 있음.
6점 = 적극적인 의미 재구성이 있으며 감정 조절 의도가 뚜렷함.
7점 = 매우 명확하고 구체적인 의미 재구성이 있고, 그 결과 감정이 완화되거나 변화한 내용까지 나타남.

억제 점수:
1점 = 억제가 전혀 없음. 감정을 숨기거나 표현을 억누른 내용이 없음.
2점 = 억제 가능성이 매우 약함. 감정을 드러내지 않았을 가능성은 있으나 불명확함.
3점 = 감정을 참으려는 표현이 약하게 있음. 하지만 구체적 행동은 부족함.
4점 = 감정을 겉으로 드러내지 않으려는 시도가 어느 정도 나타남.
5점 = 감정 표현을 의도적으로 억제한 것이 명확함.
6점 = 강한 감정을 느꼈지만 표현하지 않으려는 의도와 행동이 뚜렷함.
7점 = 강한 감정을 철저히 숨기거나 아무렇지 않은 척하는 등 억제가 매우 명확함.

[세부 판단 기준]
인지적 재평가는 다음 요소가 많고 구체적일수록 높은 점수를 부여하세요:
- 상황을 다른 관점에서 해석함
- 타인의 입장이나 상황을 고려함
- 부정적 경험을 배움, 성장, 기회로 재구성함
- 사건의 원인이나 의미를 다르게 생각함
- 재해석 후 감정이 완화되거나 변화함

억제는 다음 요소가 많고 구체적일수록 높은 점수를 부여하세요:
- 부정적 감정을 느꼈음이 명확함
- 감정을 말, 표정, 행동으로 드러내지 않음
- 티 내지 않음, 참음, 숨김, 괜찮은 척함
- 일부러 감정 표현을 억누름
- 강한 감정을 느꼈지만 겉으로는 다른 모습을 보임

[구분 규칙]
- "다르게 생각했다", "이해하려 했다", "배움이라고 생각했다"처럼 상황의 의미를 바꾸면 인지적 재평가로 평가하세요.
- "참았다", "티 내지 않았다", "아무렇지 않은 척했다"처럼 감정 표현을 숨기면 억제로 평가하세요.
- 하나의 응답에 두 전략이 모두 있으면 둘 다 점수를 부여하고 main_strategy는 mixed로 표시하세요.
- 단순히 "생각했다"라는 단어만으로 재평가로 판단하지 마세요.
- 단순히 "조용히 있었다"만으로 높은 억제 점수를 주지 마세요. 감정을 숨기거나 표현을 억제한 맥락이 필요합니다.

[출력 형식]
아래 세 단계를 순서대로 수행하세요.

Step 1 (Fact):
인지적 재평가와 억제 각각에 대해, 전사본에서 근거가 되는 표현을 직접 인용하세요.
근거가 없으면 반드시 "없음"이라고 명시하세요.

Step 2 (채점):
Step 1에서 인용한 근거를 바탕으로 각 전략의 점수를 부여하세요.
근거가 "없음"인 전략은 1~2점으로 제한하세요.

Step 3 (자기검증):
Step 2의 채점이 전사본 근거에 기반한 것인지, 아니면 추측인지 스스로 평가하세요.
"이 채점은 전사본에서 직접 확인한 근거에 기반하는가?"
- 그렇다 (True) → high 또는 medium
- 아니다 (False) → 반드시 low

그 다음 아래 JSON을 마지막에 출력하세요. JSON 이후에는 아무 텍스트도 추가하지 마세요:

{
  "confidence": ,
  "confidence_reason": ,
  "cognitive_reappraisal": ,
  "expressive_suppression": ,
  "main_strategy":
}
"""

# 실제 존재하는 참가자 번호 추출
pids = set()
for f in os.listdir(TRANSCRIPT_DIR):
    if f.endswith(".txt"):
        pid = int(f.split("_")[1])
        pids.add(pid)

print(f"총 참가자 수: {len(pids)}")

# 메인 루프
results = []

for pid in sorted(pids):
    txt_path = os.path.join(TRANSCRIPT_DIR, f"interview_{pid}_2.txt")  # 2번만 (stress 조건)

    if not os.path.exists(txt_path):
        print(f"[없음] {pid}번 전사본 없음")
        continue

    # 전사본 읽기
    with open(txt_path, "r", encoding="utf-8") as f:
        transcript = f.read()

    # GPT 호출
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"```{transcript}```"}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content
        results.append({
            "participant_id": pid,
            "gpt_output": result_text
        })
        print(f"[완료] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

# 저장
df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")
print(f"완료된 참가자 수: {len(df)}")

총 참가자 수: 75
[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료: /content/drive/MyDrive/26-1_URP/features/llm_features_deleted_yelin_lang_del.csv
완료된 참가자 수: 75


In [28]:
# CSV 불러오기
df = pd.read_csv("/content/drive/MyDrive/26-1_URP/features/llm_features_deleted_yelin_lang_del.csv")  # 파일명 확인

parsed_results = []

for _, row in df.iterrows():
    pid = row['participant_id']
    gpt_output = row['gpt_output']

    try:
        cleaned = gpt_output.replace('""', '"')
        json_match = re.search(r'\{[^{}]+\}', cleaned, re.DOTALL)

        if json_match:
            scores = json.loads(json_match.group())
            scores['participant_id'] = pid
            parsed_results.append(scores)
            print(f"[완료] {pid}번")
        else:
            print(f"[JSON 없음] {pid}번")

    except Exception as e:
        print(f"[오류] {pid}번: {e}")

df_parsed = pd.DataFrame(parsed_results)
cols = ['participant_id'] + [c for c in df_parsed.columns if c != 'participant_id']
df_parsed = df_parsed[cols]

df_parsed.to_csv(
    "/content/drive/MyDrive/26-1_URP/features/llm_features_deleted_yelin_lang_del_parsed.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n저장 완료!")
print(f"참가자 수: {len(df_parsed)}")
print(df_parsed.head())

[완료] 2번
[완료] 3번
[완료] 4번
[완료] 5번
[완료] 6번
[완료] 7번
[완료] 8번
[완료] 9번
[완료] 10번
[완료] 11번
[완료] 12번
[완료] 13번
[완료] 14번
[완료] 15번
[완료] 16번
[완료] 17번
[완료] 20번
[완료] 22번
[완료] 24번
[완료] 25번
[완료] 27번
[완료] 30번
[완료] 32번
[완료] 33번
[완료] 35번
[완료] 38번
[완료] 39번
[완료] 40번
[완료] 41번
[완료] 42번
[완료] 43번
[완료] 45번
[완료] 46번
[완료] 47번
[완료] 48번
[완료] 49번
[완료] 50번
[완료] 51번
[완료] 52번
[완료] 53번
[완료] 54번
[완료] 55번
[완료] 56번
[완료] 58번
[완료] 59번
[완료] 63번
[완료] 65번
[완료] 67번
[완료] 68번
[완료] 71번
[완료] 74번
[완료] 77번
[완료] 78번
[완료] 79번
[완료] 81번
[완료] 82번
[완료] 83번
[완료] 84번
[완료] 86번
[완료] 87번
[완료] 88번
[완료] 89번
[완료] 90번
[완료] 91번
[완료] 92번
[완료] 93번
[완료] 94번
[완료] 95번
[완료] 99번
[완료] 100번
[완료] 106번
[완료] 107번
[완료] 109번
[완료] 110번
[완료] 111번

저장 완료!
참가자 수: 75
   participant_id confidence  \
0               2       high   
1               3       high   
2               4       high   
3               5       high   
4               6       high   

                                   confidence_reason  cognitive_reappraisal  \
0  전사본에서 직접 확인한 근거에 기반하여 인지적 재평가와 억제의